# mew vs static optimisation (BioScout v3)

Same trial, same model: muscle forces, reserves and joint contact forces (JCF) from
OpenSim SO and from mew (`bioscout.simulate.mew`, muscle extrapolation web), checked against the in-vivo
hip force (OrthoLoad).

## 0. Setup

In [ ]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import bioscout as bs          # bs.Session, bs.mew.read_storage, bs.mew.contact_force_magnitude


BIOSCOUT_PATH = Path(bs.__file__).parents[1]         # repo root (bs.__file__ is bioscout/__init__.py)
EXAMPLE_DATA_PATH = BIOSCOUT_PATH / "example_data"
assert EXAMPLE_DATA_PATH.is_dir(), f"no example_data in {BIOSCOUT_PATH} (bioscout must be the C:\\Git\\bioscout checkout)"

PROJECT_DIR = EXAMPLE_DATA_PATH / "orthoload"         # has models/ + simulations/
TRIAL       = "Walking_01"

SESSION_DIR = PROJECT_DIR / "simulations" / "H1_orthoload" / "11_05_06"   # folder with session.yaml
session = bs.Session(SESSION_DIR)       # project_dir comes from session.yaml
session.describe()  


EXP         = SESSION_DIR / "2_experimental" / TRIAL                     # raw inputs + invivo_jcf.mot
ITERATION = "gpk"     # model iteration in session.yaml
SIDE      = "l"       # implanted side
REPLACE   = False     # True = overwrite the outputs of the step you run

               # per iteration: models found/MISSING, markerset, scaling, stage outputs

try:
    it = session.iteration(ITERATION)  # it.<key> = iterations.gpk.<key> (paths resolved)
except Exception as e:
    print(f"Failed to get iteration '{ITERATION}': {e}")
    it = session.add_iteration(ITERATION)
    print(f"Added iteration '{ITERATION}'")

In [ ]:
cfg       = it.trial_config(TRIAL)  # this trial's session.yaml config (same key names)
MODEL     = it.model_so                # SO / mew model: model_scaled x mvic_factor (same file at 1.0)
MASS      = session.body_mass          # session.source("body_mass") -> clickable session.yaml:line
BW        = MASS * 9.80665
TRIAL_DIR = Path(it.path) / TRIAL
MEW_DIR   = TRIAL_DIR / "mew"

# stance of the implanted leg: first on-plate contact -> next foot off
_side = {"l": "Left", "r": "Right"}[SIDE]
_ev = yaml.safe_load(open(SESSION_DIR / "session.yaml"))["trials"][TRIAL]["events"]
_fc = next(e["time"] for e in _ev if e["name"] == f"{_side} Foot Contact" and e["on_plate"])
_fo = next(e["time"] for e in _ev if e["name"] == f"{_side} Foot Off" and e["time"] > _fc)
STANCE = (_fc, _fo)

print(f"session dir: {SESSION_DIR}")
print(f"{TRIAL} | {ITERATION} | {Path(MODEL).name} | {MASS} kg | stance {STANCE[0]:.2f}-{STANCE[1]:.2f} s")

### 0.1 Add an iteration (optional)
Copies an existing iteration's block in session.yaml under a new name (config only, no results); override what differs, then run the steps below on it.

In [ ]:
# it = session.add_iteration("gpk_mri", copy_from="gpk",
#                            model_generic="../../../../../generic models/GPK_v4/GPK_v4_016_mri.osim")
session.iterations

## 1. Run
Each step is its own cell: run only what you need. Later steps reuse earlier outputs (and fill missing ones, never overwriting).

### 1.1 Scale

In [ ]:
# replace=False: skip if it.model_scaled exists. replace=True rebuilds it from
# it.model_generic + it.markerset (iterations.gpk in session.yaml); every step below uses it
it.scale_model(replace=False)


### 1.2 External biomechanics (IK -> ID)

In [ ]:
it.run(trials=[TRIAL], do_exbiomec=True, replace=REPLACE);

### 1.3 Muscle analysis (moment arms, lengths)

In [ ]:
it.run(trials=[TRIAL], do_muscle_analysis=True, replace=REPLACE);

### 1.4 Static optimisation (SO -> muscle moments -> JRA)

In [ ]:
it.run(trials=[TRIAL], do_so=True, replace=REPLACE);

### 1.5 mew (-> JRA)

In [ ]:
it.run(trials=[TRIAL], do_mew=True, replace=REPLACE);

## 2. Load

In [ ]:
FILES = {
    "SO": {"force": TRIAL_DIR / "static_optimisation" / "SO_StaticOptimization_force.sto",
           "jra":   TRIAL_DIR / "joint_contact_forces" / "Analyse_JRA_ReactionLoads_SO.sto"},
    "mew": {"force": MEW_DIR / "MuscleForces.sto",
            "jra":   MEW_DIR / "JRA_ReactionLoads.sto"},
}
COLOR = {"SO": "tab:blue", "mew": "tab:red", "in vivo": "k"}

force   = {k: bs.mew.read_storage(v["force"]) for k, v in FILES.items()}
idm     = bs.mew.read_storage(TRIAL_DIR / "external_biomechanics" / "inverse_dynamics.sto")
res_mew = bs.mew.read_storage(MEW_DIR / "Reserves.sto")
invivo  = bs.mew.read_storage(EXP / "invivo_jcf.mot")

def shade(ax):
    ax.axvspan(*STANCE, color="0.92", zorder=0)

def on_grid(df, col, t):
    return np.interp(t, df["time"], df[col])

## 3. Muscle forces by group

In [ ]:
GROUPS = {
    "Hip flexors":     ["psoas", "iliacus", "recfem", "sart", "tfl"],
    "Gluteus max":     ["glmax1", "glmax2", "glmax3"],
    "Gluteus med/min": ["glmed1", "glmed2", "glmed3", "glmin1", "glmin2", "glmin3"],
    "Adductors":       ["addbrev", "addlong", "addmagDist", "addmagIsch", "addmagMid", "addmagProx", "grac", "pect"],
    "Deep rotators":   ["piri", "quad_fem", "gemelli_sup", "gemelli_inf",
                        "obt_internus1", "obt_internus2", "obt_internus3",
                        "obt_externus1", "obt_externus2", "obt_externus3"],
    "Hamstrings":      ["bflh", "bfsh", "semimem", "semiten"],
    "Vasti":           ["vasint", "vaslat", "vasmed"],
    "Gastrocnemius":   ["gasmed", "gaslat"],
    "Soleus":          ["soleus"],
    "Dorsiflexors":    ["tibant", "edl", "ehl"],
}

fig, axs = plt.subplots(2, 5, figsize=(18, 6), sharex=True)
for ax, (group, names) in zip(axs.flat, GROUPS.items()):
    shade(ax)
    for k, df in force.items():
        cols = [f"{n}_{SIDE}" for n in names if f"{n}_{SIDE}" in df]
        ax.plot(df["time"], df[cols].sum(axis=1), color=COLOR[k], label=k)
    ax.set_title(group, fontsize=10)
axs[0, 0].set_ylabel("force (N)"); axs[1, 0].set_ylabel("force (N)")
axs[0, 0].legend(frameon=False)
fig.suptitle(f"Summed muscle force, {_side.lower()} leg (grey = stance)")
fig.tight_layout(); plt.show()

## 4. Per-muscle agreement (stance)

In [ ]:
t = force["SO"]["time"].to_numpy()
m = (t >= STANCE[0]) & (t <= STANCE[1])
rows = []
for col in force["SO"].columns:
    if not col.endswith(f"_{SIDE}") or col not in force["mew"] or "reserve" in col:
        continue
    so, mw = force["SO"][col].to_numpy()[m], on_grid(force["mew"], col, t)[m]
    rows.append({"muscle": col, "SO peak (N)": so.max(), "mew peak (N)": mw.max(),
                 "ratio mew/SO": mw.max() / so.max() if so.max() > 1 else np.nan,
                 "RMSE (N)": np.sqrt(np.mean((mw - so) ** 2)),
                 "r": np.corrcoef(so, mw)[0, 1] if so.std() and mw.std() else np.nan})
muscles = pd.DataFrame(rows).set_index("muscle").sort_values("SO peak (N)", ascending=False)
muscles.head(15).round(2)

## 5. Reserves: moment the muscles did not carry

In [ ]:
DOFS = [f"{d}_{SIDE}" for d in ("hip_flexion", "hip_adduction", "hip_rotation", "knee_angle", "ankle_angle")]

fig, axs = plt.subplots(1, len(DOFS), figsize=(18, 3.2), sharex=True)
for ax, dof in zip(axs, DOFS):
    shade(ax)
    ax.plot(idm["time"], idm[f"{dof}_moment"], color="0.4", lw=2, label="ID moment")
    if f"{dof}_reserve" in force["SO"]:
        ax.plot(force["SO"]["time"], force["SO"][f"{dof}_reserve"], color=COLOR["SO"], label="SO reserve")
    if f"{dof}_reserve" in res_mew:
        ax.plot(res_mew["time"], res_mew[f"{dof}_reserve"], color=COLOR["mew"], label="mew reserve")
    ax.set_title(dof, fontsize=10)
axs[0].set_ylabel("Nm"); axs[0].legend(frameon=False, fontsize=8)
fig.tight_layout(); plt.show()

pd.DataFrame({
    dof: {"ID peak |M| (Nm)": idm[f"{dof}_moment"].abs().max(),
          "SO reserve peak (Nm)": force["SO"].get(f"{dof}_reserve", pd.Series([np.nan])).abs().max(),
          "mew reserve peak (Nm)": res_mew.get(f"{dof}_reserve", pd.Series([np.nan])).abs().max()}
    for dof in DOFS}).T.round(1)

## 6. Joint contact forces

In [ ]:
# JRA joint names are model-specific (GPK knee = Lerner_knee_<side>)
JOINTS = {"hip": f"hip_{SIDE}", "knee": f"Lerner_knee_{SIDE}", "ankle": f"ankle_{SIDE}"}

jcf = {k: {j: bs.mew.contact_force_magnitude(v["jra"], name, MASS) for j, name in JOINTS.items()}
       for k, v in FILES.items()}
iv_t, iv_f = invivo["time"].to_numpy(), invivo["hip_measured_f"].to_numpy() / BW

fig, axs = plt.subplots(1, 3, figsize=(16, 3.8), sharex=True)
for ax, joint in zip(axs, JOINTS):
    shade(ax)
    if joint == "hip":
        ax.plot(iv_t, iv_f, color=COLOR["in vivo"], lw=2.2, label="in vivo")
    for k in FILES:
        ax.plot(*jcf[k][joint], color=COLOR[k], label=k)
    ax.set_title(f"{joint} ({JOINTS[joint]})", fontsize=10)
axs[0].set_ylabel("resultant JCF (BW)"); axs[0].legend(frameon=False)
fig.tight_layout(); plt.show()

In [ ]:
rows = []
for k in FILES:
    row = {"method": k}
    for joint, (tj, fj) in jcf[k].items():
        s = (tj >= STANCE[0]) & (tj <= STANCE[1])
        row[f"{joint} peak (BW)"] = fj[s].max()
    th, fh = jcf[k]["hip"]
    s = (th >= STANCE[0]) & (th <= STANCE[1])
    ref = np.interp(th[s], iv_t, iv_f)
    row["hip RMSE vs in vivo (BW)"] = np.sqrt(np.mean((fh[s] - ref) ** 2))
    row["hip bias (BW)"] = np.mean(fh[s] - ref)
    row["hip r"] = np.corrcoef(fh[s], ref)[0, 1]
    rows.append(row)
s = (iv_t >= STANCE[0]) & (iv_t <= STANCE[1])
rows.append({"method": "in vivo", "hip peak (BW)": iv_f[s].max()})
jcf_table = pd.DataFrame(rows).set_index("method")
jcf_table.round(2)

## 7. Save

In [ ]:
OUT = MEW_DIR / "comparison_v3"
OUT.mkdir(exist_ok=True)
muscles.to_csv(OUT / "muscle_forces_so_vs_mew.csv")
jcf_table.to_csv(OUT / "jcf_so_vs_mew.csv")
print("saved to", OUT)